In [ ]:
import os
os.chdir(path_to_wd)
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import torch
print(torch.cuda.is_available()) 

import numpy as np
import pandas as pd
import scanpy as sc
import SEACells
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import palantir
from tqdm.auto import tqdm
import pickle
from scipy.io import mmread

# Some plotting aestheticsqfree
sns.set_style('ticks')
matplotlib.rcParams['figure.figsize'] = [4, 4]
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['image.cmap'] = 'Spectral_r'

## Load data

In [ ]:
adata_full = sc.read("Reproducibility/Data/DOGMA/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")

In [ ]:
n_SEACells_dict = {
    'CD4_T': 1025,
    'CD8_T_NK_ILC': 775,
    'B': 410,
    'Myeloid': 390,
    'MSC': 113
}

In [ ]:
def run_seacells(tmp_subset, n_SEACells_dict):

    print(f"=== Running SEACells for {tmp_subset} ===")

    # -----------------------------
    # Preprocess
    # -----------------------------    
    # Subset
    adata = adata_full[adata_full.obs['lineage']==tmp_subset].copy()

    tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_latent_df.txt"
    latent_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
    latent_df = latent_df.loc[adata.obs_names, : ].values 
    adata.obsm["MultiVI_latent"] = latent_df

    tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_UMAP_df.txt"
    UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
    adata.obsm["X_umap"] = UMAP_df.values

    sc.pp.neighbors(adata, use_rep="MultiVI_latent")

    # RNA
    rna_ad = adata[:, adata.var.modality == "Gene Expression"].copy()

    # ATAC
    data_dir = os.path.expanduser(f"Reproducibility/Results/ArchR/{tmp_subset}/export/")
    counts = mmread(data_dir + "peak_counts/counts.mtx")

    cells = pd.read_csv(data_dir + "peak_counts/cells.csv", index_col=0).iloc[:, 0]
    peaks = pd.read_csv(data_dir + "peak_counts/peaks.csv", index_col=0)
    peaks.index = peaks['seqnames'] + ':' + peaks['start'].astype(str) + '-' + peaks['end'].astype(str)

    atac_ad = sc.AnnData(counts.T)
    atac_ad.obs_names = cells
    atac_ad.var_names = peaks.index

    for col in peaks.columns:
        atac_ad.var[col] = peaks[col]
    
    atac_ad.X = atac_ad.X.tocsr()

    # SVD embeddings
    atac_ad.obsm['X_svd'] = pd.read_csv(f'{data_dir}svd.csv', index_col=0).loc[atac_ad.obs_names, :].values

    # reorder CB & obs
    CB_list = [idx.rsplit("#")[1] for idx in atac_ad.obs_names]
    atac_ad.obs_names = CB_list

    # Intersection of cells
    cells_use = np.intersect1d(rna_ad.obs.index, atac_ad.obs.index)
    rna_ad = rna_ad[cells_use, :]
    atac_ad = atac_ad[cells_use, :]
    atac_ad.obs = rna_ad.obs.copy()

    # -----------------------------
    # SEACells
    # -----------------------------
    model = SEACells.core.SEACells(
        adata,
        build_kernel_on="MultiVI_latent",
        use_gpu=True,
        n_SEACells=n_SEACells_dict[tmp_subset],
        n_waypoint_eigs=10,
        convergence_epsilon=1e-5,
    )

    model.construct_kernel_matrix()
    model.initialize_archetypes()

    SEACells.plot.plot_initialization(rna_ad, model)
    model.fit()

    # evaluation
    model.plot_convergence()
    SEACells.plot.plot_2D(rna_ad, key="X_umap", colour_metacells=False)
    SEACells.plot.plot_SEACell_sizes(rna_ad, bins=5)

    compactness = SEACells.evaluate.compactness(rna_ad, "MultiVI_latent")
    SEACell_purity = SEACells.evaluate.compute_celltype_purity(rna_ad, "celltype")
    separation = SEACells.evaluate.separation(rna_ad, "MultiVI_latent", nth_nbr=1)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)
    sns.boxplot(data=compactness, y="compactness", ax=axes[0]); axes[0].set_title("Compactness"); sns.despine(ax=axes[0])
    sns.boxplot(data=SEACell_purity, y="celltype_purity", ax=axes[1]); axes[1].set_title("Celltype Purity"); sns.despine(ax=axes[1])
    sns.boxplot(data=separation, y="separation", ax=axes[2]); axes[2].set_title("Separation"); sns.despine(ax=axes[2])
    plt.tight_layout(); plt.show()

    # -----------------------------
    # Metacell matrices
    # -----------------------------
    atac_ad.obs["SEACell"] = rna_ad.obs["SEACell"]

    rna_ad.layers["counts"] = rna_ad.X
    atac_ad.layers["counts"] = atac_ad.X
    rna_meta_ad = SEACells.core.summarize_by_SEACell(rna_ad, SEACells_label="SEACell", summarize_layer="counts")
    atac_meta_ad = SEACells.core.summarize_by_SEACell(atac_ad, SEACells_label="SEACell", summarize_layer="counts")

    # assign top cell type
    top_ct = atac_ad.obs["celltype"].groupby(atac_ad.obs["SEACell"]).value_counts().groupby(level=0, group_keys=False).head(1)
    rna_meta_ad.obs["celltype"] = top_ct[rna_meta_ad.obs_names].index.get_level_values(1)
    atac_meta_ad.obs["celltype"] = top_ct[atac_meta_ad.obs_names].index.get_level_values(1)

    # -----------------------------
    # Save
    # -----------------------------
    dir_path = f"Reproducibility/Results/SEACells/{tmp_subset}"
    os.makedirs(dir_path, exist_ok=True)

    rna_meta_ad.write(f"{dir_path}/UC_DOGMA_SEACells_rna_meta_ad.h5ad")
    atac_meta_ad.write(f"{dir_path}/UC_DOGMA_SEACells_atac_meta_ad.h5ad")

    rna_ad.obs.to_csv(f"{dir_path}/UC_DOGMA_{tmp_subset}_metadata_w_SEACells.txt", sep ='\t')
    rna_meta_ad.obs.to_csv(f"{dir_path}/UC_DOGMA_SEACells_metacell_metadata.txt", sep="\t")

    print(f"Finished SEACells for {tmp_subset}, results saved to {dir_path}")


In [ ]:
run_seacells('CD4_T', n_SEACells_dict)
run_seacells('CD8_T_NK_ILC', n_SEACells_dict)
run_seacells('B', n_SEACells_dict)
run_seacells('Myeloid', n_SEACells_dict)
run_seacells('MSC', n_SEACells_dict)